In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and Quick Quality Assessment of Data

In [31]:
df = pd.read_csv('data/bank-additional-full.csv', sep=';')

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

In [33]:
# Display summary statistics for each column
for col in df.columns:
    print(f"{col} " + "=" * 50)
    if df[col].dtype == 'str':
        print(df[col].value_counts())
    elif df[col].dtype in ['int64', 'float64']:
        print(df[col].describe())
        print("Missing values:", df[col].isnull().sum())
    print()

age ==================================================
count    41188.00000
mean        40.02406
std         10.42125
min         17.00000
25%         32.00000
50%         38.00000
75%         47.00000
max         98.00000
Name: age, dtype: float64
Missing values: 0

job ==================================================
job
admin.           10422
blue-collar       9254
technician        6743
services          3969
management        2924
retired           1720
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
unknown            330
Name: count, dtype: int64

marital ==================================================
marital
married     24928
single      11568
divorced     4612
unknown        80
Name: count, dtype: int64

education ==================================================
education
university.degree      12168
high.school             9515
basic.9y                6045
professional.course     5243
basic.4y         

In [34]:
# missing values
missing_values = (df == 'unknown').sum()
percent_missing = ((missing_values / len(df)) * 100).round(2)
missing_df = pd.DataFrame({'Missing Values': missing_values, 'Percent Missing': percent_missing})
print(missing_df[missing_df['Missing Values'] > 0])

rows_with_missing = ((df == 'unknown') | df.isnull()).any(axis=1).sum()
percent_missing = (rows_with_missing / len(df)) * 100
print(f"\nNumber of rows with missing values: {rows_with_missing} ({percent_missing:.2f}%)")

           Missing Values  Percent Missing
job                   330             0.80
marital                80             0.19
education            1731             4.20
default              8597            20.87
housing               990             2.40
loan                  990             2.40

Number of rows with missing values: 10700 (25.98%)


In [35]:
df['y'].value_counts() / len(df) * 100

y
no     88.734583
yes    11.265417
Name: count, dtype: float64

## Handling Missing Values

**SUMMARY**
- test for MCAR, MNAR
- multinomial logistic regression for nominal features
- KNN imputation for ordinal features

# Preprocessing

**Imputation Summary**
- treat `unknown` (string) as missing value
- treat missing values for `default` as a category
- test for MCAR, MNAR
- multinomial logistic regression for nominal features
- KNN imputation for ordinal features

**Transformation Summary**
- transform `y` to boolean
- drop `duration`, `pdays` column
- one-hot encode `job`, `marital`, `education`, `contact`, `month`, and `day_of_week`
- ordinal encode `education`
- multicollinearity check
  - drop `poutcome_nonexistent` due to high correlation with `previous`
  - perform PCA for macroeconomic indicators (`cons.price.id`, `euribor3m`, `nr.employed`, `euribor3m`)

## Test for MCAR, MNAR

In [36]:
from lib import *

In [37]:
mcar_test_results = tp_test_mcar(df)

In [38]:
missing_cols = ["job", "marital", "education", "default", "housing", "loan"]
significants = mcar_test_results[
    (mcar_test_results['missing_column'] != 'default')          # treat missing default as a separate category
    & (mcar_test_results['p_value'] < 0.05)                     # significant association with tested column
    & (~mcar_test_results['tested_column'].isin(missing_cols))  # exclude columns with missing values as tested columns
].sort_values('missing_column')
print(significants['missing_column'].unique())
significants
# From this, we can see that all missing columns are not MCAR

<StringArray>
['education', 'housing', 'job', 'loan', 'marital']
Length: 5, dtype: str


,missing_column,tested_column,p_value
29,education,y,1.629895e-05
89,education,nr.employed,1.048714e-03
88,education,euribor3m,4.915301e-03
87,education,cons.conf.idx,9.867530e-06
25,education,contact,3.416578e-05
26,education,month,6.544704e-19
27,education,day_of_week,8.552996e-04
28,education,poutcome,7.100547e-04
80,education,age,9.500015e-40
86,education,cons.price.idx,2.793088e-11


## Imputation

In [39]:
from lib import *
from sklearn.model_selection import train_test_split

# split into train/test before imputation to avoid data leakage
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [46]:
# take ~4 mins
logreg_opts = {
    'solver': 'newton-cg',
    'max_iter': 1000,
}
multi_imputer = TPMultinomialImputer(logreg_opts=logreg_opts)
multi_imputer.fit(df_train)

In [47]:
df_train_multi_imputed = multi_imputer.transform(df_train, verbose=True)


Imputation summary for 'job':
               Before  Percent Before   After  Imputed  Percent Imputed
job                                                                    
admin.           8328           25.27  8420.0     92.0             0.35
blue-collar      7439           22.58  7559.0    120.0             0.46
technician       5352           16.24  5365.0     13.0             0.05
services         3212            9.75  3212.0      0.0             0.00
management       2310            7.01  2310.0      0.0             0.00
retired          1363            4.14  1395.0     32.0             0.12
self-employed    1153            3.50  1153.0      0.0             0.00
entrepreneur     1145            3.47  1145.0      0.0             0.00
housemaid         867            2.63   867.0      0.0             0.00
unemployed        798            2.42   798.0      0.0             0.00
student           721            2.19   726.0      5.0             0.02
unknown           262            

In [48]:
df_test_multi_imputed = multi_imputer.transform(df_test, verbose=True)


Imputation summary for 'job':
               Before  Percent Before   After  Imputed  Percent Imputed
job                                                                    
admin.           2094           25.42  2116.0     22.0             0.32
blue-collar      1815           22.03  1850.0     35.0             0.51
technician       1391           16.89  1393.0      2.0             0.03
services          757            9.19   757.0      0.0             0.00
management        614            7.45   614.0      0.0             0.00
retired           357            4.33   364.0      7.0             0.10
entrepreneur      311            3.78   311.0      0.0             0.00
self-employed     268            3.25   268.0      0.0             0.00
unemployed        216            2.62   216.0      0.0             0.00
housemaid         193            2.34   193.0      0.0             0.00
student           154            1.87   156.0      2.0             0.03
unknown            68            

In [49]:
knn_imputer = TPKNNImputer()
knn_imputer.fit(df_train)

In [ ]:
df_train_knn_imputed = knn_imputer.transform(df_train_multi_imputed, verbose=True)

In [ ]:
df_test_knn_imputed = knn_imputer.transform(df_test_multi_imputed, verbose=True)

## Transformations

In [52]:
df_train_transformed = tp_simple_transform(df_train_knn_imputed)
df_test_transformed = tp_simple_transform(df_test_knn_imputed)

In [53]:
df_train_encoded = tp_encode(df_train_transformed)
df_test_encoded = tp_encode(df_test_transformed)

In [54]:
# checkpoint
df_train_encoded.to_csv('data/df_train_encoded.csv', index=False)
df_test_encoded.to_csv('data/df_test_encoded.csv', index=False)

## Handle Multicollinearity

In [ ]:
# multicollinearity check
correlation_matrix = df_train_encoded.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

In [56]:
"""
From this we can see that 
- 4 macroeconomic indicators are highly correlated with each other. ('cons.price.id', 'euribor3m', 'nr.employed', 'euribor3m')
- 'marital_single' and 'marital_married' are highly correlated. Will leave as is.
- 'poutcome_nonexistent' and 'previous' are highly correlated. Will drop 'poutcome_nonexistent'.
"""

# check for high correlation (> 0.7)
high_correlations = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > 0.7: # type: ignore
            high_correlations.append((
                correlation_matrix.columns[i], 
                correlation_matrix.columns[j], 
                correlation_matrix.iloc[i, j])
            )

df_high_corr = pd.DataFrame(high_correlations, columns=['col1', 'col2', 'correlation'])
df_high_corr

,col1,col2,correlation
0,cons.price.idx,emp.var.rate,0.773785
1,euribor3m,emp.var.rate,0.972084
2,nr.employed,emp.var.rate,0.906548
3,nr.employed,euribor3m,0.945144
4,marital_single,marital_married,-0.775600
5,poutcome_nonexistent,previous,-0.880837
6,poutcome_success,previously_contacted,0.950187


In [ ]:
df_train_encoded.drop(columns=['previously_contacted'], inplace=True)
df_test_encoded.drop(columns=['previously_contacted'], inplace=True)